In [1]:
using PowerModels, DataFrames, CSV, LinearAlgebra, SparseArrays

function DCOPF_Constraints_Enhanced(case_path::String, output_dir::String)
    """
    增强版DCOPF约束导出 - 额外保存支路首末节点信息
    用于GNN图结构构建
    """
    
    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]
    
    data = parse_file(case_path)
    ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]
    
    # generator limits
    gen = sort(collect(keys(ref[:gen])))
    gen_pmin = [ref[:gen][i]["pmin"] for i in gen]
    gen_pmax = [ref[:gen][i]["pmax"] for i in gen]
    df_gen_limits = DataFrame(gen_id=gen, pgmin=gen_pmin, pgmax=gen_pmax)
    CSV.write(joinpath(output_dir, "$(case_name)_gen_limits.csv"), df_gen_limits)

    # ========== 新增：保存支路首末节点信息 ==========
    branch = sort(collect(keys(ref[:branch])))
    branch_rate_a = [get(ref[:branch][i], "rate_a", Inf) for i in branch]
    
    # 提取支路的首末节点
    branch_f_bus = [ref[:branch][i]["f_bus"] for i in branch]
    branch_t_bus = [ref[:branch][i]["t_bus"] for i in branch]
    branch_r = [ref[:branch][i]["br_r"] for i in branch]
    branch_x = [ref[:branch][i]["br_x"] for i in branch]
    
    df_branch_info = DataFrame(
        branch_id=branch,
        f_bus=branch_f_bus,
        t_bus=branch_t_bus,
        r_pu=branch_r,
        x_pu=branch_x,
        rate_a=branch_rate_a
    )
    CSV.write(joinpath(output_dir, "$(case_name)_branch_info.csv"), df_branch_info)
    # ===============================================
    
    # 保留原有的branch_limits（向后兼容）
    df_branch_limits = DataFrame(branch_id=branch, rate_a=branch_rate_a)
    CSV.write(joinpath(output_dir, "$(case_name)_branch_limits.csv"), df_branch_limits)
    
    bus = sort(collect(keys(ref[:bus])))
    bus_lookup = Dict(bus_id => i for (i, bus_id) in enumerate(bus))
    bus_count = length(bus)

    # cost
    cost_c2, cost_c1, cost_c0 = [], [], []
    for gen_id in gen
        cost_coeffs = ref[:gen][gen_id]["cost"]
        if length(cost_coeffs) == 3 
            push!(cost_c2, cost_coeffs[1])
            push!(cost_c1, cost_coeffs[2])
            push!(cost_c0, cost_coeffs[3])
        elseif length(cost_coeffs) == 2 
            push!(cost_c2, 0.0) 
            push!(cost_c1, cost_coeffs[1])
            push!(cost_c0, cost_coeffs[2])
        else 
            push!(cost_c2, 0.0)
            push!(cost_c1, 0.0)
            push!(cost_c0, 0.0)
        end
    end
    df_gen_costs = DataFrame(gen_id=gen, cost_c2=cost_c2, cost_c1=cost_c1, cost_c0=cost_c0)
    CSV.write(joinpath(output_dir, "$(case_name)_gen_costs.csv"), df_gen_costs)
    
    # Bbus matrix
    Bbus = zeros(bus_count, bus_count)
    for (i, branch) in ref[:branch]
        f_bus = bus_lookup[branch["f_bus"]]
        t_bus = bus_lookup[branch["t_bus"]]
        b = -1 / branch["br_x"] 
        Bbus[f_bus, t_bus] += b
        Bbus[t_bus, f_bus] += b
        Bbus[f_bus, f_bus] -= b
        Bbus[t_bus, t_bus] -= b
    end
    
    for (i, bus) in ref[:bus]
        bus_idx = bus_lookup[i]
        Bbus[bus_idx, bus_idx] += get(bus, "bs", 0.0)
    end

    # Branch-Bus matrix (A) and Branch Susceptance matrix 
    branch_count = length(branch)
    A = spzeros(Int, branch_count, bus_count)
    b_diag = spzeros(Float64, branch_count, branch_count)
    for (i, br_id) in enumerate(branch)
        branch = ref[:branch][br_id]
        f_bus = bus_lookup[branch["f_bus"]]
        t_bus = bus_lookup[branch["t_bus"]]
        A[i, f_bus] = 1
        A[i, t_bus] = -1
        b_diag[i, i] = -1 / branch["br_x"]
    end
    
    # PTDF matrix
    slack_bus_idx = bus_lookup[first(collect(keys(ref[:ref_buses])))]
    non_slack_indices = [i for i in 1:bus_count if i != slack_bus_idx]
    
    Bbus_ns = Bbus[non_slack_indices, non_slack_indices]
    A_ns = A[:, non_slack_indices]
    
    ptdf_matrix_ns = b_diag * A_ns * inv(Matrix(Bbus_ns))
    
    ptdf_matrix = zeros(branch_count, bus_count)
    ptdf_matrix[:, non_slack_indices] = ptdf_matrix_ns

    df_ptdf = DataFrame(ptdf_matrix, :auto)
    CSV.write(joinpath(output_dir, "$(case_name)_ptdf_matrix.csv"), df_ptdf)
    
    # Bus-Generator Map
    gen_count = length(gen)
    bus_gen_map = zeros(Int, bus_count, gen_count)
    for (i, gen_id) in enumerate(gen)
        gen_bus = ref[:gen][gen_id]["gen_bus"]
        bus_pos = bus_lookup[gen_bus]
        bus_gen_map[bus_pos, i] = 1
    end
    df_bus_gen_map = DataFrame(bus_gen_map, :auto)
    CSV.write(joinpath(output_dir, "$(case_name)_bus_gen_map.csv"), df_bus_gen_map)

    # Bus ID mapping (sorted bus IDs → matrix row index)
    # Critical for case300 where bus IDs are non-contiguous (e.g. 250,281,...,9533)
    df_bus_ids = DataFrame(bus_id=bus)
    CSV.write(joinpath(output_dir, "$(case_name)_bus_ids.csv"), df_bus_ids)

    # base_mva
    df_base_mva = DataFrame(parameter=["base_mva"], value=[ref[:baseMVA]])
    CSV.write(joinpath(output_dir, "$(case_name)_base_mva.csv"), df_base_mva)
    
    println("✓ Enhanced DCOPF constraints exported (including branch topology)")
end

# Main 
case_file = raw"C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case39_epri.m"
output_dir = raw"C:\Users\Aloha\Desktop\dataset\DCOPF Constraints\case39"
DCOPF_Constraints_Enhanced(case_file, output_dir)

[info | PowerModels]: removing 1 cost terms from generator 8: [3155.0181, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 4: [3484.4642999999996, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 1: [672.4778, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 5: [2465.2994, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 2: [1470.7625, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 6: [3230.6483, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 7: [1815.7477, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 10: [2743.4444, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 9: [2250.3168, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 3: [2480.4734, 0.0]
✓ Enhanced DCOPF constraints exported (including branch topology)


In [1]:
#用于毕业设计-数据中心调频时空调度
using PowerModels, DataFrames, CSV, LinearAlgebra, SparseArrays

function DCOPF_Constraints_Enhanced(case_path::String, output_dir::String)
    """
    增强版DCOPF约束导出 - 额外保存支路首末节点信息和节点负荷
    用于GNN图结构构建以及调频时空调度模型
    """
    
    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]
    
    data = parse_file(case_path)
    ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]
    
    # generator limits
    gen = sort(collect(keys(ref[:gen])))
    gen_pmin = [ref[:gen][i]["pmin"] for i in gen]
    gen_pmax = [ref[:gen][i]["pmax"] for i in gen]
    df_gen_limits = DataFrame(gen_id=gen, pgmin=gen_pmin, pgmax=gen_pmax)
    CSV.write(joinpath(output_dir, "$(case_name)_gen_limits.csv"), df_gen_limits)

    # ========== 支路首末节点信息 ==========
    branch = sort(collect(keys(ref[:branch])))
    branch_rate_a = [get(ref[:branch][i], "rate_a", Inf) for i in branch]
    
    branch_f_bus = [ref[:branch][i]["f_bus"] for i in branch]
    branch_t_bus = [ref[:branch][i]["t_bus"] for i in branch]
    branch_r = [ref[:branch][i]["br_r"] for i in branch]
    branch_x = [ref[:branch][i]["br_x"] for i in branch]
    
    df_branch_info = DataFrame(
        branch_id=branch,
        f_bus=branch_f_bus,
        t_bus=branch_t_bus,
        r_pu=branch_r,
        x_pu=branch_x,
        rate_a=branch_rate_a
    )
    CSV.write(joinpath(output_dir, "$(case_name)_branch_info.csv"), df_branch_info)
    
    # 保留原有的branch_limits（向后兼容）
    df_branch_limits = DataFrame(branch_id=branch, rate_a=branch_rate_a)
    CSV.write(joinpath(output_dir, "$(case_name)_branch_limits.csv"), df_branch_limits)
    
    bus = sort(collect(keys(ref[:bus])))
    bus_lookup = Dict(bus_id => i for (i, bus_id) in enumerate(bus))
    bus_count = length(bus)

    # ========== 节点负荷 (PD, QD) ==========
    # 注意: ref[:load] 是真正存储负荷的字典, 一个 bus 上可能挂多个 load,
    # 所以使用 += 累加; 单位从 pu 转换为 MW/MVar.
    bus_pd = zeros(bus_count)
    bus_qd = zeros(bus_count)
    for (load_id, load) in ref[:load]
        bus_idx = bus_lookup[load["load_bus"]]
        bus_pd[bus_idx] += load["pd"] * ref[:baseMVA]
        bus_qd[bus_idx] += load["qd"] * ref[:baseMVA]
    end
    df_bus_load = DataFrame(
        bus_id  = bus,
        pd_mw   = bus_pd,
        qd_mvar = bus_qd
    )
    CSV.write(joinpath(output_dir, "$(case_name)_bus_load.csv"), df_bus_load)
    println("  → bus_load.csv: total PD = ", round(sum(bus_pd), digits=2), " MW")
    
    # cost
    cost_c2, cost_c1, cost_c0 = [], [], []
    for gen_id in gen
        cost_coeffs = ref[:gen][gen_id]["cost"]
        if length(cost_coeffs) == 3 
            push!(cost_c2, cost_coeffs[1])
            push!(cost_c1, cost_coeffs[2])
            push!(cost_c0, cost_coeffs[3])
        elseif length(cost_coeffs) == 2 
            push!(cost_c2, 0.0) 
            push!(cost_c1, cost_coeffs[1])
            push!(cost_c0, cost_coeffs[2])
        else 
            push!(cost_c2, 0.0)
            push!(cost_c1, 0.0)
            push!(cost_c0, 0.0)
        end
    end
    df_gen_costs = DataFrame(gen_id=gen, cost_c2=cost_c2, cost_c1=cost_c1, cost_c0=cost_c0)
    CSV.write(joinpath(output_dir, "$(case_name)_gen_costs.csv"), df_gen_costs)
    
    # Bbus matrix
    Bbus = zeros(bus_count, bus_count)
    for (i, branch) in ref[:branch]
        f_bus = bus_lookup[branch["f_bus"]]
        t_bus = bus_lookup[branch["t_bus"]]
        b = -1 / branch["br_x"] 
        Bbus[f_bus, t_bus] += b
        Bbus[t_bus, f_bus] += b
        Bbus[f_bus, f_bus] -= b
        Bbus[t_bus, t_bus] -= b
    end
    
    for (i, bus) in ref[:bus]
        bus_idx = bus_lookup[i]
        Bbus[bus_idx, bus_idx] += get(bus, "bs", 0.0)
    end

    # Branch-Bus matrix (A) and Branch Susceptance matrix 
    branch_count = length(branch)
    A = spzeros(Int, branch_count, bus_count)
    b_diag = spzeros(Float64, branch_count, branch_count)
    for (i, br_id) in enumerate(branch)
        branch = ref[:branch][br_id]
        f_bus = bus_lookup[branch["f_bus"]]
        t_bus = bus_lookup[branch["t_bus"]]
        A[i, f_bus] = 1
        A[i, t_bus] = -1
        b_diag[i, i] = -1 / branch["br_x"]
    end
    
    # PTDF matrix
    slack_bus_idx = bus_lookup[first(collect(keys(ref[:ref_buses])))]
    non_slack_indices = [i for i in 1:bus_count if i != slack_bus_idx]
    
    Bbus_ns = Bbus[non_slack_indices, non_slack_indices]
    A_ns = A[:, non_slack_indices]
    
    ptdf_matrix_ns = b_diag * A_ns * inv(Matrix(Bbus_ns))
    
    ptdf_matrix = zeros(branch_count, bus_count)
    ptdf_matrix[:, non_slack_indices] = ptdf_matrix_ns

    df_ptdf = DataFrame(ptdf_matrix, :auto)
    CSV.write(joinpath(output_dir, "$(case_name)_ptdf_matrix.csv"), df_ptdf)
    
    # Bus-Generator Map
    gen_count = length(gen)
    bus_gen_map = zeros(Int, bus_count, gen_count)
    for (i, gen_id) in enumerate(gen)
        gen_bus = ref[:gen][gen_id]["gen_bus"]
        bus_pos = bus_lookup[gen_bus]
        bus_gen_map[bus_pos, i] = 1
    end
    df_bus_gen_map = DataFrame(bus_gen_map, :auto)
    CSV.write(joinpath(output_dir, "$(case_name)_bus_gen_map.csv"), df_bus_gen_map)

    # Bus ID mapping (sorted bus IDs → matrix row index)
    df_bus_ids = DataFrame(bus_id=bus)
    CSV.write(joinpath(output_dir, "$(case_name)_bus_ids.csv"), df_bus_ids)

    # base_mva
    df_base_mva = DataFrame(parameter=["base_mva"], value=[ref[:baseMVA]])
    CSV.write(joinpath(output_dir, "$(case_name)_base_mva.csv"), df_base_mva)
    
    println("✓ Enhanced DCOPF constraints exported (including branch topology & bus load)")
end

# Main 
case_file = raw"C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case39_epri.m"
output_dir = raw"C:\Users\Aloha\Desktop\dataset\DCOPF Constraints\case39"
DCOPF_Constraints_Enhanced(case_file, output_dir)

[info | PowerModels]: removing 1 cost terms from generator 8: [3155.0181, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 4: [3484.4642999999996, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 1: [672.4778, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 5: [2465.2994, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 2: [1470.7625, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 6: [3230.6483, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 7: [1815.7477, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 10: [2743.4444, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 9: [2250.3168, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 3: [2480.4734, 0.0]
  → bus_load.csv: total PD = 6254.23 MW
✓ Enhanced DCOPF constraints exported (including branch topology & bus load)
